In [1]:
# imports and env
import base64
from dotenv import load_dotenv
from langchain_unstructured.document_loaders import UnstructuredLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import PydanticOutputParser
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

/var/folders/xc/lny8rntd7nn0g08bxqjc_s000000gn/T/ipykernel_72405/1379007190.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores.utils import filter_complex_metadata


False

In [2]:
# load PDF
PDF_PATH = "../data/crag_paper.pdf"

loader = UnstructuredLoader(
    PDF_PATH,
    mode="elements",
    strategy="hi_res",
    extract_images_in_pdf=True,
)
elements = loader.load()

print(f"Loaded {len(elements)} elements")
for cat in sorted(set(el.metadata.get("category", "unknown") for el in elements)):
    count = sum(1 for el in elements if el.metadata.get("category") == cat)
    print(f"  {cat}: {count}")

INFO: pikepdf C++ to Python logger bridge initialized
INFO: HTTP Request: HEAD https://huggingface.co/unstructuredio/yolo_x_layout/resolve/main/yolox_l0.05.onnx "HTTP/1.1 302 Found"
INFO: Reading PDF for file: ../data/crag_paper.pdf ...


Loaded 268 elements
  FigureCaption: 6
  Footer: 1
  Formula: 1
  Header: 1
  Image: 8
  ListItem: 41
  NarrativeText: 107
  Table: 7
  Title: 32
  UncategorizedText: 64


In [11]:
s = [s.metadata["category"] for s in elements]
# print(set(s))
e = [element for element in elements if element.metadata["category"]=="Image "]
e[0]

IndexError: list index out of range

In [3]:
for element in elements:
    if element.metadata.get("category") == "Image":
        print(element.metadata.keys())
        print(element.metadata)
        break

dict_keys(['source', 'coordinates', 'last_modified', 'filetype', 'languages', 'page_number', 'image_path', 'file_directory', 'filename', 'category', 'element_id'])
{'source': '../data/crag_paper.pdf', 'coordinates': {'points': ((np.float64(372.93637515555537), np.float64(2949.874959546666)), (np.float64(372.93637515555537), np.float64(3083.2151141477775)), (np.float64(470.7191490011109), np.float64(3083.2151141477775)), (np.float64(470.7191490011109), np.float64(2949.874959546666))), 'system': 'PixelSpace', 'layout_width': 2894, 'layout_height': 4093}, 'last_modified': '2026-06-15T13:00:31', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'image_path': '/Users/itaakash/Desktop/Lakhan/Tasks/ragAdancedCampusX/Advanced_Rag_Codes-main/12_rag_multimodal/notebook/figures/figure-1-1.jpg', 'file_directory': '../data', 'filename': 'crag_paper.pdf', 'category': 'Image', 'element_id': '0eda6d21a6f5ee087ef5f507aaa8896c'}


In [4]:
# caption images with VLM
vlm = ChatOllama(model="qwen3.5:9b")

IMAGE_CAPTION_SYSTEM_PROMPT = """You are a document analysis assistant. Your task is to generate \
detailed, accurate descriptions of images extracted from a document. These descriptions will be \
embedded into a vector store and used for semantic retrieval, so they must capture all information \
a user might search for.

For each image, describe:
- The image type (chart, diagram, photograph, table, illustration, screenshot, etc.)
- All visible text, labels, titles, captions, and annotations
- Key data, values, trends, or patterns (especially for charts and graphs)
- The main subject and all important visual elements
- Spatial relationships and structure where relevant

Be specific and thorough. Avoid vague language."""

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def caption_image(image_path: str) -> str:
    b64 = encode_image(image_path)
    messages = [
        SystemMessage(content=IMAGE_CAPTION_SYSTEM_PROMPT),
        HumanMessage(content=[
            {"type": "text", "text": "Describe this image extracted from a document."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]),
    ]
    return vlm.invoke(messages).content

In [5]:
image_docs = []

for el in elements:
    if el.metadata.get("category") == "Image":
        image_path = el.metadata.get("image_path", "")
        if image_path:
            caption = caption_image(image_path)
            image_docs.append(Document(page_content=caption, metadata=el.metadata))

print(f"Captioned {len(image_docs)} images")

INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Captioned 8 images


In [10]:

e = [element for element in elements if element.metadata["category"]=="Image "]
e[0]
print(image_docs[-1].page_content)

IndexError: list index out of range

In [12]:
# split text
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

text_elements = [el for el in elements if el.metadata.get("category") != "Image"]

text_docs = splitter.split_documents(text_elements)
caption_docs = splitter.split_documents(image_docs)

all_docs = text_docs + caption_docs
print(f"Total chunks: {len(all_docs)} ({len(text_docs)} text + {len(caption_docs)} captions)")

Total chunks: 291 (269 text + 22 captions)


In [13]:
# embeddings
embeddings = OllamaEmbeddings(model="embeddinggemma")

In [14]:
# vector store
vector_store = Chroma.from_documents(filter_complex_metadata(all_docs), embeddings)

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


In [15]:
# retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [ ]:
# RAG chain
prompt = ChatPromptTemplate.from_messages([
    ("human", "Answer the question based only on the following context:\n\n{context}\n\nQuestion: {question}"),
])

def build_messages(inputs):
    docs = inputs["docs"]
    question = inputs["question"]

    text_chunks = [d for d in docs if d.metadata.get("category") != "Image"]
    image_chunks = [d for d in docs if d.metadata.get("category") == "Image"]

    text_context = "\n\n".join(d.page_content for d in text_chunks)

    messages = prompt.format_messages(context=text_context, question=question)

    if image_chunks:
        seen_paths = set()
        image_content = []
        for doc in image_chunks:
            image_path = doc.metadata.get("image_path")
            if image_path and image_path not in seen_paths:
                seen_paths.add(image_path)
                image_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{encode_image(image_path)}"},
                })

        if image_content:
            text_content = messages[-1].content
            messages[-1] = HumanMessage(content=[
                {"type": "text", "text": text_content},
                *image_content,
            ])

    return messages

llm = ChatOllama(model="qwen3.5:9b")

chain = (
    {"docs": retriever, "question": RunnablePassthrough()}
    | RunnableLambda(build_messages)
    | {"response": llm, "context": RunnablePassthrough()}
)

In [18]:
# test
question = "How does Self-CRAG compares with Self-RAG as shown in the line chart. Can you explain this in a little bit more detail?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Based on the line chart provided, here is how Self-CRAG compares to Self-RAG:

**Consistent Superiority**
The grey line with diamond markers (representing **Self-CRAG**) is positioned consistently higher than the green line with star markers (representing **Self-RAG**) across the entire graph. This indicates that for any given level of retrieval accuracy, Self-CRAG achieves a higher "Accuracy of generation."

**Performance Stability vs. Decline**
*   **At High Retrieval Accuracy (Left side):** When retrieval accuracy is high (e.g., at 69.8), Self-CRAG performs significantly better than Self-RAG. The chart shows Self-CRAG starting near 62% accuracy while Self-RAG starts around 55%.
*   **At Low Retrieval Accuracy (Right side):** As the retrieval accuracy drops (moving from left to right, down to 10), both models see their generation accuracy decline. However, Self-RAG suffers a steep drop, falling to below 35%. In contrast, Self-CRAG remains much more stable, staying above 50% even at t

In [19]:
len(answer["context"][0].content)

2

In [20]:
question = "Computational requirements of CRAG vs Self-RAG and which was has faster execution time and can you give me the actual TFLOPS values?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Based on the provided context:

*   **TFLOPs per token (Computational requirements):** CRAG is **27.2**, while Self-RAG ranges from **26.5 to 132.4**.
*   **Execution time:** CRAG is **0.512 seconds**, while Self-RAG is **0.741 seconds**.
*   **Faster Execution Time:** **CRAG** has the faster execution time (0.512s compared to 0.741s).
